# Guarantee full coverage from the offset-optimized regular grid

`04_sweep_reduced_fov_path_combinations_40x.ipynb` found `regular +
optimize_offset` leaves real tissue uncovered (175.85 um^2) at 40X on the
LT066_sample_01/merfish boundary -- something never seen at 60X on the
same boundary. This notebook diagnoses the actual root cause and
validates the fix.

**Root cause found**: `create_grid_positions` shifts the grid's centre
(`cx`/`cy`) by the requested *offset*, but sized each axis's point count
(`n`, via `_spaced_coords`) from the boundary's ORIGINAL (pre-shift) bbox
span alone. A shifted centre can sit closer to one bbox edge than the
other -- if the boundary's own span has little "slack" past an exact
multiple of `step_size`, a large enough offset then leaves the outermost
point short of the far edge. Fixed in `positions.py`: each axis's point
count is now sized from the LARGER of its two post-shift half-spans
(`rx`/`ry`), which exactly reduces to the original behaviour at
`offset=(0, 0)` (both halves equal `span/2`) -- zero regression risk for
the default, most-common case.


## 1 — Setup


In [1]:
import os
import sys
import shutil
from pathlib import Path

import numpy as np
from shapely.geometry import box as shapely_box
from shapely.ops import unary_union

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent   # MERci/
SAMPLE_DIR = MERCI_DIR.parent
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.configs   import get_fov_geometry
from MERci.acquisition.positions import (
    load_boundary_polygon, load_hole_polygons,
    create_grid_positions, generate_scanning_path, filter_scanning_path,
    build_reduced_fov_path,
)

NOTEBOOK_NAME = "05_guarantee_offset_grid_coverage"
DATA_DIR      = MERCI_DIR / "cache" / "tests" / "create_positions" / NOTEBOOK_NAME / "data" / "boundary"
print(f"DATA_DIR: {DATA_DIR}")


DATA_DIR: /n/home06/lsepulvedaduran/251225_LT027_saving_time/MERci/cache/tests/create_positions/05_guarantee_offset_grid_coverage/data/boundary


## 2 — Load the same real boundary as notebooks 02/04

Own independent local copy, per `NOTEBOOK_GUIDELINES.md`'s portability
convention.


In [2]:
SOURCE_BOUNDARY_DIR = Path(
    "/n/holylfs05/LABS/zhuang_lab/Lab/shared/projects/lineage_tracing/experiments/"
    "LT066_sample_01/merfish/positions/boundaries/from_mosaic"
)
if not DATA_DIR.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for f in sorted(SOURCE_BOUNDARY_DIR.glob("*.txt")):
        shutil.copy2(f, DATA_DIR / f.name)
    print(f"Copied {len(list(DATA_DIR.glob('*.txt')))} boundary/hole file(s).")
else:
    print(f"Using existing local copy: {DATA_DIR}")

boundary_polygon = load_boundary_polygon(DATA_DIR / "boundary_positions.txt")
hole_polygons    = load_hole_polygons(DATA_DIR)
tissue           = boundary_polygon.difference(unary_union(hole_polygons))

xmin, ymin, xmax, ymax = boundary_polygon.bounds
print(f"Tissue area: {tissue.area / 1e6:.2f} mm^2")
print(f"bbox span x: {xmax - xmin:.1f} um   y: {ymax - ymin:.1f} um")


Using existing local copy: /n/home06/lsepulvedaduran/251225_LT027_saving_time/MERci/cache/tests/create_positions/05_guarantee_offset_grid_coverage/data/boundary
Tissue area: 30.60 mm^2
bbox span x: 9010.0 um   y: 6185.0 um


## 3 — Reproduce the bug at 40X (the objective that first exposed it)


In [3]:
pixel_size_um, image_size_px = get_fov_geometry("ST2", "40X")
FOV_SIZE_UM  = pixel_size_um * image_size_px
STEP_SIZE_UM = FOV_SIZE_UM * 0.9
half = FOV_SIZE_UM / 2.0

print(f"span_x / step: {(xmax - xmin) / STEP_SIZE_UM:.3f}")
print(f"span_y / step: {(ymax - ymin) / STEP_SIZE_UM:.3f}   (close to a whole number -> little slack)")

def old_buggy_create_grid_positions(boundary_polygon, step_size, direction="vertical", offset=(0.0, 0.0)):
    # Historical reproduction of the ORIGINAL (buggy) construction, kept
    # here ONLY to demonstrate the bug this notebook fixes -- positions.py
    # itself no longer has this version (see its git history). The single
    # difference from the current, fixed positions.py is that xs/ys are
    # sized from the boundary's own (pre-shift) xmin/xmax/ymin/ymax here,
    # not from the post-shift half-spans.
    from MERci.acquisition.positions import _spaced_coords
    xmin, ymin, xmax, ymax = boundary_polygon.bounds
    cx = (xmin + xmax) / 2.0 + offset[0]
    cy = (ymin + ymax) / 2.0 + offset[1]
    if direction == "vertical":
        xs = _spaced_coords(cx, xmin, xmax, step_size, even=True)
        ys = _spaced_coords(cy, ymin, ymax, step_size, even=False)
    else:
        xs = _spaced_coords(cx, xmin, xmax, step_size, even=False)
        ys = _spaced_coords(cy, ymin, ymax, step_size, even=True)
    Xg, Yg = np.meshgrid(xs, ys)
    return np.stack([Xg, Yg], axis=-1), xs, ys

def uncovered_for(grid_fn, offset):
    grid, _, _ = grid_fn(boundary_polygon, STEP_SIZE_UM, offset=offset)
    path = generate_scanning_path(grid)
    filtered = filter_scanning_path(path, boundary_polygon, hole_polygons, FOV_SIZE_UM)
    boxes = [shapely_box(x - half, y - half, x + half, y + half) for x, y in filtered]
    return len(filtered), tissue.difference(unary_union(boxes)).area

# The specific offset that first exposed this bug in
# create_positions/04_sweep_reduced_fov_path_combinations_40x.ipynb --
# hardcoded (not re-derived via optimize_grid_offset) because that
# function now uses the FIXED create_grid_positions internally, so it no
# longer picks an offset that triggers the old bug.
original_offset = (45.51551999999998, -75.85919999999999)
print(f"\nOffset that originally exposed the bug: {original_offset}")

n_old, unc_old = uncovered_for(old_buggy_create_grid_positions, original_offset)
n_new, unc_new = uncovered_for(create_grid_positions, original_offset)
print(f"OLD buggy create_grid_positions : n={n_old:4d}  uncovered={unc_old:8.2f} um^2")
print(f"CURRENT fixed create_grid_positions : n={n_new:4d}  uncovered={unc_new:8.2f} um^2")


span_x / step: 32.992
span_y / step: 22.648   (close to a whole number -> little slack)

Offset that originally exposed the bug: (45.51551999999998, -75.85919999999999)


OLD buggy create_grid_positions : n= 530  uncovered=  175.09 um^2
CURRENT fixed create_grid_positions : n= 533  uncovered=    0.00 um^2


## 4 — Stress test: every offset in a dense grid, both objectives

Not just the one offset `optimize_grid_offset` happened to pick -- sweep
a dense `25x25` offset grid (finer than `optimize_grid_offset`'s own
default `9x9`) and confirm the worst case across ALL of them is fully
covered, for both the 40X objective that exposed the bug and the 60X
objective notebook 02 uses (same real boundary, different step size).


In [4]:
def worst_uncovered(fov_size_um, step_size_um, n_samples=25):
    half_ = fov_size_um / 2.0
    offsets = np.linspace(-step_size_um / 2.0, step_size_um / 2.0, n_samples, endpoint=False)
    worst = 0.0
    for dx in offsets:
        for dy in offsets:
            grid, _, _ = create_grid_positions(boundary_polygon, step_size_um, offset=(dx, dy))
            path = generate_scanning_path(grid)
            filtered = filter_scanning_path(path, boundary_polygon, hole_polygons, fov_size_um)
            boxes = [shapely_box(x - half_, y - half_, x + half_, y + half_) for x, y in filtered]
            worst = max(worst, tissue.difference(unary_union(boxes)).area)
    return worst

px_40x, sz_40x = get_fov_geometry("ST2", "40X")
fov_40x = px_40x * sz_40x
step_40x = fov_40x * 0.9
worst_40x = worst_uncovered(fov_40x, step_40x)
print(f"40X: worst uncovered area over 25x25 offsets = {worst_40x:.4f} um^2")

px_60x, sz_60x = get_fov_geometry("ST2", "60X")
fov_60x = px_60x * sz_60x
step_60x = 182.06208   # same measured value notebook 02 uses
worst_60x = worst_uncovered(fov_60x, step_60x)
print(f"60X: worst uncovered area over 25x25 offsets = {worst_60x:.4f} um^2")


40X: worst uncovered area over 25x25 offsets = 0.0000 um^2


60X: worst uncovered area over 25x25 offsets = 0.0000 um^2


## 5 — Re-validate `build_reduced_fov_path(irregular_grid=False, optimize_offset=True)`


In [5]:
for name, fov, step in (("40X", fov_40x, step_40x), ("60X", fov_60x, step_60x)):
    res = build_reduced_fov_path(boundary_polygon, hole_polygons, step, fov,
                                  optimize_offset=True, remove_redundant_fovs=False)
    print(f"{name}: n_fovs={len(res.coords):4d}  uncovered={res.redundant.uncovered_area_um2:.4f} um^2")


40X: n_fovs= 532  uncovered=0.0000 um^2


60X: n_fovs=1123  uncovered=0.0000 um^2


## 6 — Takeaways

- **Root cause**: `create_grid_positions` sized each axis's point count
  from the boundary's pre-shift bbox span, while centring the lattice at
  a post-shift centre -- a real, previously-undocumented mismatch that
  can leave the outermost point short of the far bbox edge whenever the
  boundary's own span has little slack past a `step_size` multiple and
  the searched offset is large. This boundary's y-span sits at `22.6-32.9`
  step-size multiples (close to whole numbers, i.e. little slack)
  depending on objective, which is exactly why it only showed up at 40X
  (different `step_size` -> different slack) and not 60X, on the *same*
  boundary.

- **Fix**: size each axis from the larger of its two post-shift
  half-spans instead. Confirmed 0.0 um^2 uncovered for the specific
  offset that exposed the bug, AND across a dense 25x25 offset sweep at
  both 40X and 60X -- not just the one offset that happened to trigger it.

- **`build_reduced_fov_path(optimize_offset=True)` is now coverage-safe
  at both objectives** on this real boundary.
